In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/data/Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

display(df)

In [0]:
print(df.columns)

In [0]:
df.printSchema()

In [0]:
clean_df = df.dropDuplicates()
clean_df = clean_df.na.drop()

display(clean_df) 

In [0]:
from pyspark.sql.functions import col

clean_df = clean_df.toDF(
    *[c.replace(" ", "_") for c in clean_df.columns]
)

print(clean_df.columns)

In [0]:
clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("superstore_master")

In [0]:
display(spark.table("superstore_master"))

In [0]:
incremental_df = clean_df.limit(5)

display(incremental_df)

In [0]:
incremental_df = clean_df.limit(3)

display(incremental_df)

In [0]:
SHOW TABLES

In [0]:
spark.catalog.listTables()

In [0]:
from pyspark.sql.functions import when, col

incremental_df = incremental_df.withColumn(
    "Customer_Name",
    when(col("Row_ID") == 5, "Updated Customer")
    .otherwise(col("Customer_Name"))
)

display(incremental_df)

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(
    spark,
    "superstore_master"
)

deltaTable.alias("target").merge(
    incremental_df.alias("source"),
    "target.Row_ID = source.Row_ID"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Merge completed successfully maza agyaaaa")

In [0]:
display(
    spark.sql("""
    SELECT Row_ID, Customer_Name
    FROM superstore_master
    WHERE Row_ID = 5
    """)
)

In [0]:
print("Master Row Count:",
      spark.table("superstore_master").count())
print("compeleted :)")      

In [0]:
display(spark.table("superstore_master"))